# 01. Первые числа

Первый шаг проекта. Цель вечера — получить три числа и понять, что данные вообще живые:

1. сколько в магазине клиентов;
2. какая доля возвращается на второй месяц;
3. сколько выручки приносят лучшие 10%.

Скучную часть — чтение файла, распознавание колонок, очистку — уже делает пакет
`src/analytics`. Здесь мы им пользуемся и занимаемся аналитикой.

**Ноутбук запускается сразу**: если файла с реальными данными нет, он сгенерирует
похожий синтетический и всё посчитает на нём. Так можно проверить, что код работает,
ещё до скачивания датасета.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Ноутбук лежит в notebooks/, а код аналитики — в backend/src/.
# Чтобы импорт `from src.analytics ...` заработал, добавляем в пути поиска
# именно папку backend: Python ищет пакет `src` внутри неё.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
BACKEND = ROOT / "backend"
if str(BACKEND) not in sys.path:
    sys.path.insert(0, str(BACKEND))

from src.analytics import assess, guess_mapping, normalize, read_table
from src.analytics.quality import drop_duplicates, trim_incomplete_tail
from src.analytics.schema import AMOUNT, CUSTOMER_ID, ORDER_DATE

pd.set_option("display.width", 120)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")
plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

print("Корень проекта:", ROOT)
print("Код аналитики: ", BACKEND / "src" / "analytics")

## Шаг 1. Данные

Рабочий датасет — **UCI Online Retail II**: около миллиона транзакций британского
интернет-магазина за два года. В нём есть всё, что бывает в реальных выгрузках:
возвраты, оптовые заказы, пропущенные идентификаторы клиентов.

Скачать: <https://archive.ics.uci.edu/dataset/502/online+retail+ii>

Положить файл в `data/raw/online_retail_II.xlsx` — эта папка не попадает в git.
Пока файла нет, ячейка ниже сделает синтетические данные.

In [ ]:
DATA_PATH = ROOT / "data" / "raw" / "online_retail_II.xlsx"


def make_synthetic(n_customers: int = 1500, seed: int = 42) -> pd.DataFrame:
    """Выгрузка, похожая на то, что присылает небольшой магазин.

    Колонки названы по-русски и «грязно» — чтобы проверить автоматическое
    распознавание: суммы строками с пробелом в разрядах, даты day-first.
    """
    rng = np.random.default_rng(seed)
    start, end = pd.Timestamp("2023-01-01"), pd.Timestamp("2024-12-31")
    rows = []

    for customer in range(n_customers):
        moment = start + pd.Timedelta(days=int(rng.integers(0, 600)))
        # Пуассон даёт реалистичную картину: большинство купит один-два раза,
        # небольшая часть — много.
        for _ in range(1 + rng.poisson(1.8)):
            if moment > end:
                break
            rows.append(
                {
                    "Дата заказа": moment.strftime("%d.%m.%Y"),
                    "Код клиента": f"C-{customer:05d}",
                    "Номер чека": f"R-{len(rows):06d}",
                    "Сумма, руб": f"{rng.lognormal(7.3, 0.7):,.2f}".replace(",", " "),
                }
            )
            moment += pd.Timedelta(days=int(rng.gamma(2.0, 30.0)) + 1)

    frame = pd.DataFrame(rows)
    # Немного возвратов — без них картина слишком чистая.
    refunds = frame.sample(frac=0.02, random_state=seed).copy()
    refunds["Сумма, руб"] = "-" + refunds["Сумма, руб"]
    return pd.concat([frame, refunds], ignore_index=True)


if DATA_PATH.exists():
    raw = read_table(DATA_PATH)
    source = "Online Retail II"
else:
    raw = make_synthetic()
    source = "синтетика (реальный файл не найден)"

print(f"Источник: {source}")
print(f"Строк: {len(raw):,}".replace(",", " "))
raw.head()

## Шаг 2. Распознавание колонок

Пользователь назовёт колонки как захочет. `guess_mapping` предлагает соответствие,
а человек в интерфейсе его подтверждает или правит — именно так это будет работать
в продукте.

In [ ]:
mapping = guess_mapping(raw)

for canonical, column in mapping.as_dict().items():
    print(f"{canonical:<12} <- {column}")

if mapping.unmapped:
    print("\nНе распознано:", ", ".join(map(str, mapping.unmapped)))

assert mapping.is_valid, f"Не хватает колонок: {mapping.missing_required}"

## Шаг 3. Очистка

`normalize` приводит типы и выбрасывает строки без даты, клиента или суммы.
Дальше убираем полные дубли и, если последний месяц явно неполный, отрезаем его:
иначе на графике будет ложный обвал продаж в конце.

In [ ]:
clean = normalize(raw, mapping)
clean, duplicates = drop_duplicates(clean)
clean, tail_trimmed = trim_incomplete_tail(clean)

print(f"Осталось строк: {len(clean):,}".replace(",", " "))
print(f"Удалено дублей: {duplicates}")
print(f"Неполный последний месяц отрезан: {'да' if tail_trimmed else 'нет'}")
clean.head()

## Шаг 4. Отчёт о качестве

Это первый экран будущего продукта. Он показывает, что сервис действительно
посмотрел в файл, и честно говорит, чего на этих данных посчитать нельзя.

In [ ]:
report = assess(clean, rows_raw=len(raw))
print(report.summary())

if report.warnings:
    print("\nНа что обратить внимание:")
    for warning in report.warnings:
        print(f"  • {warning}")

print("\nЧто можно считать на этих данных:")
titles = {
    "rfm": "RFM-сегментация",
    "cohorts": "Когортный анализ",
    "clv": "Прогноз LTV",
    "churn": "Модель оттока",
}
for key, is_available in report.available.items():
    print(f"  {'+' if is_available else '-'} {titles[key]}")

## Шаг 5. Три числа

Ради них всё и затевалось.

In [ ]:
frame = clean.copy()
frame["month"] = frame[ORDER_DATE].dt.to_period("M")

# ── 1. Клиенты ────────────────────────────────────────────────────────
customers_total = frame[CUSTOMER_ID].nunique()

# ── 2. Возврат на второй месяц ────────────────────────────────────────
first_month = frame.groupby(CUSTOMER_ID)["month"].min()
frame["cohort"] = frame[CUSTOMER_ID].map(first_month)
frame["age"] = (frame["month"] - frame["cohort"]).apply(lambda delta: delta.n)

# Тех, кто пришёл в последнем месяце, не учитываем: у них ещё не было
# возможности вернуться, иначе мы занизим удержание.
last_month = frame["month"].max()
eligible = set(first_month[first_month < last_month].index)

returned = frame.loc[
    (frame["age"] == 1) & frame[CUSTOMER_ID].isin(eligible), CUSTOMER_ID
].nunique()
retention_m2 = returned / len(eligible) if eligible else 0.0

# ── 3. Доля выручки лучших 10% ────────────────────────────────────────
by_customer = frame.groupby(CUSTOMER_ID)[AMOUNT].sum().sort_values(ascending=False)
top_size = max(1, round(len(by_customer) * 0.10))
top_share = by_customer.head(top_size).sum() / by_customer.sum()

print(f"Клиентов:                      {customers_total:,}".replace(",", " "))
print(f"Возвращаются на второй месяц:  {retention_m2:.1%}")
print(f"Доля выручки лучших 10%:       {top_share:.1%}")

**Как читать эти числа.**

Удержание второго месяца ниже 10% означает, что бизнес живёт на постоянном
привлечении новых людей, и работа с оттоком даст много. Доля выручки лучших 10%
выше 50% говорит, что база сильно расслоена и терять такого клиента особенно дорого —
именно на этом будет строиться ценность продукта.

In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(13, 4))

monthly = frame.groupby("month")[AMOUNT].sum()
left.bar([str(p) for p in monthly.index], monthly.to_numpy(), color="#2563eb")
left.set_title("Выручка по месяцам")
left.tick_params(axis="x", rotation=90, labelsize=7)

# Кривая концентрации: сколько выручки накапливают первые N% клиентов.
cumulative = by_customer.cumsum() / by_customer.sum()
share_of_customers = np.arange(1, len(cumulative) + 1) / len(cumulative)
right.plot(share_of_customers * 100, cumulative.to_numpy() * 100, color="#2563eb")
right.plot([0, 100], [0, 100], "--", color="#9ca3af", label="равномерно")
right.axvline(10, color="#dc2626", linestyle=":", label="лучшие 10%")
right.set_title("Концентрация выручки")
right.set_xlabel("% клиентов")
right.set_ylabel("% выручки")
right.legend()

plt.tight_layout()
plt.show()

## Что дальше

Этот ноутбук закрывает первый пункт фазы 2 из `ROADMAP.md`. Следующие — уже
самостоятельная работа, и это самая интересная часть проекта:

- `02_rfm.ipynb` — сегментация по давности, частоте и сумме покупок. Квантили по
  каждой из трёх метрик, затем осмысленные названия сегментов и доля выручки в каждом.
- `03_cohorts.ipynb` — матрица «месяц первой покупки × месяц жизни». Заготовка уже
  есть: колонки `cohort` и `age` посчитаны выше, остаётся свести их в `pivot_table`.
- `04_btyd.ipynb` — BG/NBD и Gamma-Gamma, прогноз LTV.
- `05_churn.ipynb` — модель оттока со срезами по времени.

Когда код в ноутбуке перестаёт меняться каждый вечер, он переезжает в `src/analytics/`
и обрастает тестами. Так ядро продукта собирается по кусочку, а не переписывается
целиком в конце.